# Fine-Tuning DistilBERT for Sentiment Analysis on IMDB Dataset

In [2]:
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
import torch
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

print("=" * 70)
print("FINE-TUNING SENTIMENT CLASSIFICATION - IMDB")
print("=" * 70)

FINE-TUNING SENTIMENT CLASSIFICATION - IMDB


### PART 1: LOAD DATASET


In [3]:

print("\nLoading IMDB dataset...")
dataset = load_dataset("imdb")

print(f"Dataset loaded:")
print(f"   Train: {len(dataset['train']):,} examples")
print(f"   Test:  {len(dataset['test']):,} examples")

# View an example
print(f"\nExample:")
print(f"   Text: {dataset['train'][0]['text'][:200]}...")
print(f"   Label: {dataset['train'][0]['label']} (0=Negative, 1=Positive)")



Loading IMDB dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Dataset loaded:
   Train: 25,000 examples
   Test:  25,000 examples

Example:
   Text: I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ev...
   Label: 0 (0=Negative, 1=Positive)


### PART 2: LOAD MODEL

In [4]:
print("\nLoading the model...")
model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2  # 2 classes: Negative/Positive
)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

print(f"Model loaded on {device}")



Loading the model...


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded on cuda


### PART 3: PREPARE DATA AND TRAINING

In [5]:
print("\nTokenizing the dataset...")

def preprocess(examples):
    """Convert text to numerical inputs"""
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )

# Tokenize dataset
tokenized_dataset = dataset.map(preprocess, batched=True)

# Optional: take a subset for faster training
train_dataset = tokenized_dataset["train"].shuffle(seed=42).select(range(5000))
test_dataset = tokenized_dataset["test"].shuffle(seed=42).select(range(1000))

print(f"Data prepared:")
print(f"   Train: {len(train_dataset):,} examples")
print(f"   Test:  {len(test_dataset):,} examples")

# Evaluation metrics
def compute_metrics(eval_pred):
    """Compute accuracy, precision, recall, F1"""
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)

    accuracy = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average='binary'
    )

    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }

# Training configuration
print("\nConfiguring training...")

training_args = TrainingArguments(
    output_dir="./imdb_model",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_steps=100,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    report_to="none"
)

# Create Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

print("Trainer configured")


Tokenizing the dataset...


Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

Data prepared:
   Train: 5,000 examples
   Test:  1,000 examples

Configuring training...
Trainer configured


### PART 4: TRAIN THE MODEL

In [6]:
print("\nSTARTING TRAINING")
print("=" * 70)

trainer.train()
print("\nTraining completed!")


STARTING TRAINING


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.309900,0.303858,0.873000,0.811744,0.963115,0.880975
2,0.183000,0.294506,0.899000,0.865784,0.938525,0.900688
3,0.110800,0.340509,0.903000,0.884086,0.922131,0.902708



Training completed!


### PART5 EVALUATE ON TEST SET

In [7]:
print("\nEvaluation on test set...")
results = trainer.evaluate()

print("\nFINAL RESULTS")
print("=" * 70)
print(f"   Accuracy:  {results['eval_accuracy']:.2%}")
print(f"   Precision: {results['eval_precision']:.2%}")
print(f"   Recall:    {results['eval_recall']:.2%}")
print(f"   F1 Score:  {results['eval_f1']:.2%}")

# Save model
print("\nSaving the model...")
trainer.save_model("./imdb_model_final")
tokenizer.save_pretrained("./imdb_model_final")
print("Model saved in ./imdb_model_final")


Evaluation on test set...



FINAL RESULTS
   Accuracy:  89.90%
   Precision: 86.58%
   Recall:    93.85%
   F1 Score:  90.07%

Saving the model...
Model saved in ./imdb_model_final


### PART 6 PREDICT ON NEW TEXTS

In [8]:
def predict_sentiment(text):
    """Predict sentiment of a text"""
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
        predictions = torch.softmax(outputs.logits, dim=1)
        predicted_class = torch.argmax(predictions, dim=1).item()
        confidence = predictions[0][predicted_class].item()
    sentiment = "Positive" if predicted_class == 1 else "Negative"
    return sentiment, confidence

# Test new texts
test_texts = [
    "This movie was absolutely amazing! I loved every minute of it.",
    "Terrible film, waste of time. Do not watch.",
    "It was okay, nothing special but not bad either.",
    "Best movie I've seen this year! Highly recommended!",
    "Boring and predictable. Very disappointed."
]

for text in test_texts:
    sentiment, confidence = predict_sentiment(text)
    print(f"\n\"{text[:60]}...\"")
    print(f"   → {sentiment} (confidence: {confidence:.1%})")

print("\nFINISHED")
print("=" * 70)



"This movie was absolutely amazing! I loved every minute of i..."
   → Positive (confidence: 99.2%)

"Terrible film, waste of time. Do not watch...."
   → Negative (confidence: 98.7%)

"It was okay, nothing special but not bad either...."
   → Negative (confidence: 62.2%)

"Best movie I've seen this year! Highly recommended!..."
   → Positive (confidence: 98.5%)

"Boring and predictable. Very disappointed...."
   → Negative (confidence: 97.8%)

FINISHED
